# MCP Demo

LLM tool calls forwarded to the live msp_search MCP server.

Requires the server running at `http://127.0.0.1:8325/mcp`.

In [1]:
import sys
from pathlib import Path
from typing import Any
from fastmcp import Client

sys.path.insert(0, str(Path().resolve()))
from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.llm_logger import AsyncLLMLogger
MCP_URL = "http://127.0.0.1:8325/mcp"

## Build tools + registry from MCP server

In [2]:
async with Client(MCP_URL) as client:
    mcp_tools = await client.list_tools()

def to_openai_tool(t) -> dict:
    schema = {k: v for k, v in (t.inputSchema or {}).items() if k != "additionalProperties"}
    return {"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": schema}}

def make_handler(name: str):
    async def handler(args: dict[str, Any]) -> str:
        async with Client(MCP_URL) as client:
            result = await client.call_tool(name, args)
            return result.data if isinstance(result.data, str) else str(result.data)
    return handler

TOOLS         = [to_openai_tool(t) for t in mcp_tools]
TOOL_REGISTRY = {t.name: make_handler(t.name) for t in mcp_tools}

print("Tools:", [t["function"]["name"] for t in TOOLS])

Tools: ['get_current_date_time', 'fetch_and_convert', 'search_web']


## Create server + model

In [3]:
PROVIDER = "ollama"
MODEL    = "gpt-oss:20b"
# PROVIDER = "lm_studio"
# MODEL    = "google/gemma-4-e4b"

server = LLMProviderPool("providers.example.yaml")
llm    = server.load_model(PROVIDER, MODEL)
print(f"{PROVIDER} / {MODEL}")

ollama / gpt-oss:20b


## Demo — get current date & time

In [4]:
result = await llm.call(
    messages=[{"role": "user", "content": "What is the current date and time? Use the tool."}],
    tools=TOOLS,
    tool_registry=TOOL_REGISTRY,
    think=True,
    options={"max_tokens": 200},
)
print(result)

The current date and time is **2026‑05‑16 19:19:32**.


## Demo — web search

In [5]:
result = await llm.call(
    messages=[{"role": "user", "content": "Search for 'Python 3.13 new features' and give a one-sentence summary."}],
    tools=TOOLS,
    tool_registry=TOOL_REGISTRY,
    think=True,
    options={"max_tokens": 500},
)
print(result)

**No – Python 3.13.0 has already been superseded.**

The most recent 3.13 release is  
**Python 3.13.13**, which was released on **7 April 2026**.  

That version incorporates all the changes introduced in 3.13.0 (plus a dozen additional bug‑fixes, build improvements, and documentation updates). The 3.13 series is now in **maintenance** mode; the next feature‑release series is Python 3.14.
